# เซิร์ฟเวอร์ AI สำหรับบอทคุยในห้องเสียง

โน้ตบุ๊กนี้เปิด AI ทั้งสามส่วนที่บอทต้องใช้ไว้บน Colab แล้วสร้างลิงก์ให้บอทเรียกใช้ได้

| ส่วน | ใช้อะไร |
| --- | --- |
| ฟังเสียง → ข้อความ | faster-whisper (รันบน GPU ของ Colab) |
| คิดคำตอบ | Typhoon 2.5 ผ่าน Ollama (รันบน GPU ของ Colab) |
| ข้อความ → เสียงพูด | edge-tts เสียงภาษาไทยของ Microsoft (ส่งข้อความไปเซิร์ฟเวอร์ Microsoft) |

**วิธีใช้**
1. เมนู **Runtime → Change runtime type** เลือก GPU (T4 ก็พอ)
2. เมนู **Runtime → Run all** แล้วรอจนเซลล์สุดท้ายพิมพ์สองบรรทัด `OPENAI_BASE_URL=...` และ `OPENAI_API_KEY=...`
3. คัดลอกสองบรรทัดนั้นไปแทนที่ในไฟล์ `.env` ของบอท แล้วรันบอทใหม่
4. เล่นเสร็จแล้วเลือก **Runtime → Disconnect and delete runtime** เพื่อหยุดใช้หน่วยประมวลผล

ลิงก์จะเปลี่ยนทุกครั้งที่เปิดโน้ตบุ๊กใหม่ ถ้าตั้ง `API_KEY` ไว้คงที่ จะต้องแก้แค่ `OPENAI_BASE_URL`

> ข้อควรระวัง: ข้อตกลงของ Colab ห้ามใช้ runtime เปิดเป็นบริการเว็บ การใช้แบบนี้มีความเสี่ยงที่ runtime จะถูกตัดหรือบัญชีถูกจำกัด

## 1. ตั้งค่า

In [ ]:
# โมเดลคิดคำตอบ (ชื่อที่ Ollama ดึงได้) ตัวเริ่มต้นเล็กพอสำหรับ T4
# ถ้าใช้ GPU L4 หรือ A100 ลองตัวใหญ่ที่ฉลาดกว่า: "hf.co/typhoon-ai/typhoon2.5-qwen3-30b-a3b-gguf:Q4_K_M"
LLM_MODEL = "hf.co/typhoon-ai/typhoon2.5-qwen3-4b-gguf"

# โมเดลฟังเสียง: "large-v3" แม่นสุด, "large-v3-turbo" เร็วกว่าแต่ภาษาไทยพลาดบ่อยกว่า
WHISPER_MODEL = "large-v3"
STT_LANGUAGE = "th"

# เสียงพูด: "th-TH-PremwadeeNeural" (ผู้หญิง) หรือ "th-TH-NiwatNeural" (ผู้ชาย)
TTS_VOICE = "th-TH-PremwadeeNeural"

# รหัสผ่านที่บอทใช้เรียกเซิร์ฟเวอร์นี้ เว้นว่างไว้เพื่อสุ่มใหม่ทุกครั้ง
API_KEY = ""

import secrets
API_KEY = API_KEY or secrets.token_urlsafe(24)

## 2. ติดตั้งโปรแกรม

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!pip install -q faster-whisper edge-tts fastapi uvicorn httpx python-multipart
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

## 3. เปิด Ollama และดาวน์โหลดโมเดลคิดคำตอบ

In [ ]:
import subprocess, time, urllib.request

ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama ไม่ยอมเปิด ดู ollama.log")

!ollama pull {LLM_MODEL}

## 4. โค้ดเซิร์ฟเวอร์

In [ ]:
%%writefile server.py
"""OpenAI-compatible voice AI server for the Discord bot.

Serves the three endpoints the bot calls, so the bot only needs
OPENAI_BASE_URL and OPENAI_API_KEY pointed here:

  POST /v1/audio/transcriptions  speech -> text   (faster-whisper)
  POST /v1/chat/completions      reply            (proxied to Ollama)
  POST /v1/audio/speech          text -> mp3      (edge-tts)

Every /v1 request must carry "Authorization: Bearer <API_KEY>".
"""

import asyncio
import json
import os
import secrets
import tempfile
from contextlib import asynccontextmanager

import edge_tts
import httpx
from fastapi import Depends, FastAPI, File, Form, HTTPException, Request, UploadFile
from fastapi.responses import JSONResponse, Response, StreamingResponse
from faster_whisper import WhisperModel

API_KEY = os.environ["API_KEY"]
LLM_MODEL = os.environ["LLM_MODEL"]
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")
WHISPER_MODEL = os.getenv("WHISPER_MODEL", "large-v3")
WHISPER_DEVICE = os.getenv("WHISPER_DEVICE", "cuda")
WHISPER_COMPUTE = os.getenv("WHISPER_COMPUTE", "float16")
STT_LANGUAGE = os.getenv("STT_LANGUAGE", "th")
TTS_VOICE = os.getenv("TTS_VOICE", "th-TH-PremwadeeNeural")

whisper: WhisperModel | None = None
whisper_lock = asyncio.Lock()  # one transcription at a time on the GPU


@asynccontextmanager
async def lifespan(_app: FastAPI):
    global whisper
    # Load at startup so a bad model name or missing CUDA fails before the tunnel opens.
    whisper = WhisperModel(WHISPER_MODEL, device=WHISPER_DEVICE, compute_type=WHISPER_COMPUTE)
    yield


app = FastAPI(lifespan=lifespan)


def require_key(request: Request) -> None:
    given = request.headers.get("authorization", "")
    if not secrets.compare_digest(given.encode(), f"Bearer {API_KEY}".encode()):
        raise HTTPException(status_code=401, detail="invalid API key")


@app.get("/health")
async def health():
    return {"ok": True, "llm": LLM_MODEL, "whisper": WHISPER_MODEL, "voice": TTS_VOICE}


# ---- Speech to text ----
@app.post("/v1/audio/transcriptions", dependencies=[Depends(require_key)])
async def transcriptions(
    file: UploadFile = File(...),
    model: str = Form("whisper-1"),  # accepted for compatibility; WHISPER_MODEL is used
    language: str | None = Form(None),
):
    suffix = os.path.splitext(file.filename or "")[1] or ".mp3"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(await file.read())
        path = tmp.name
    try:
        async with whisper_lock:
            text = await asyncio.to_thread(_transcribe, path, language or STT_LANGUAGE)
    finally:
        os.unlink(path)
    return {"text": text}


def _transcribe(path: str, language: str) -> str:
    # The VAD filter drops silence, which is where Whisper invents phrases like "thanks for watching".
    segments, _info = whisper.transcribe(path, language=language, beam_size=5, vad_filter=True)
    return "".join(s.text for s in segments).strip()


# ---- Text to speech ----
@app.post("/v1/audio/speech", dependencies=[Depends(require_key)])
async def speech(request: Request):
    body = await request.json()
    text = (body.get("input") or "").strip()
    if not text:
        raise HTTPException(status_code=400, detail="input is empty")
    # The bot sends OpenAI voice names like "alloy"; only Edge voice names ("...Neural") are used as-is.
    voice = body.get("voice") or ""
    if "Neural" not in voice:
        voice = TTS_VOICE
    audio = bytearray()
    async for chunk in edge_tts.Communicate(text, voice).stream():
        if chunk["type"] == "audio":
            audio.extend(chunk["data"])
    return Response(content=bytes(audio), media_type="audio/mpeg")


# ---- Chat ----
class ThinkStripper:
    """Removes <think>...</think> blocks from streamed text, even when a tag is split across chunks."""

    OPEN, CLOSE = "<think>", "</think>"

    def __init__(self):
        self.buf = ""
        self.inside = False

    def feed(self, text: str) -> str:
        self.buf += text
        out = []
        while True:
            tag = self.CLOSE if self.inside else self.OPEN
            i = self.buf.find(tag)
            if i >= 0:
                if not self.inside:
                    out.append(self.buf[:i])
                self.buf = self.buf[i + len(tag):]
                self.inside = not self.inside
                continue
            # Hold back a trailing fragment that could be the start of the tag.
            keep = next((k for k in range(min(len(tag) - 1, len(self.buf)), 0, -1) if tag.startswith(self.buf[-k:])), 0)
            if not self.inside:
                out.append(self.buf[: len(self.buf) - keep])
            self.buf = self.buf[len(self.buf) - keep:]
            return "".join(out)

    def flush(self) -> str:
        rest = "" if self.inside else self.buf
        self.buf = ""
        return rest


def _sse(payload: dict) -> str:
    return f"data: {json.dumps(payload, ensure_ascii=False)}\n\n"


@app.post("/v1/chat/completions", dependencies=[Depends(require_key)])
async def chat(request: Request):
    body = await request.json()
    body["model"] = LLM_MODEL  # whatever OPENAI_MODEL the bot sends, use the model Ollama has loaded
    url = f"{OLLAMA_URL}/v1/chat/completions"

    if not body.get("stream"):
        async with httpx.AsyncClient(timeout=300) as client:
            res = await client.post(url, json=body)
        if res.status_code != 200:
            return JSONResponse(status_code=502, content={"error": res.text})
        data = res.json()
        for choice in data.get("choices", []):
            msg = choice.get("message") or {}
            if isinstance(msg.get("content"), str):
                stripper = ThinkStripper()
                msg["content"] = (stripper.feed(msg["content"]) + stripper.flush()).strip()
        return data

    async def relay():
        stripper = ThinkStripper()
        last = None
        async with httpx.AsyncClient(timeout=300) as client:
            async with client.stream("POST", url, json=body) as res:
                if res.status_code != 200:
                    error = (await res.aread()).decode(errors="replace")
                    yield _sse({"error": {"message": f"Ollama returned {res.status_code}: {error}"}})
                    return
                async for line in res.aiter_lines():
                    if not line.startswith("data:"):
                        continue
                    data = line[5:].strip()
                    if data == "[DONE]":
                        break
                    try:
                        chunk = json.loads(data)
                    except json.JSONDecodeError:
                        continue
                    last = chunk
                    for choice in chunk.get("choices", []):
                        delta = choice.get("delta") or {}
                        delta.pop("reasoning", None)
                        if isinstance(delta.get("content"), str):
                            delta["content"] = stripper.feed(delta["content"])
                    yield _sse(chunk)
        rest = stripper.flush()
        if rest and last:
            yield _sse({**last, "choices": [{"index": 0, "delta": {"content": rest}, "finish_reason": None}]})
        yield "data: [DONE]\n\n"

    return StreamingResponse(relay(), media_type="text/event-stream")

## 5. เปิดเซิร์ฟเวอร์

ครั้งแรกต้องดาวน์โหลดโมเดลฟังเสียงด้วย อาจใช้เวลาสักครู่

In [ ]:
import os, glob

# faster-whisper ต้องหาไลบรารี cuDNN/cuBLAS ที่มากับ PyTorch ให้เจอ
lib_dirs = glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/*/lib")
env = {
    **os.environ,
    "LD_LIBRARY_PATH": ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")]),
    "API_KEY": API_KEY,
    "LLM_MODEL": LLM_MODEL,
    "WHISPER_MODEL": WHISPER_MODEL,
    "STT_LANGUAGE": STT_LANGUAGE,
    "TTS_VOICE": TTS_VOICE,
}
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"],
    env=env, stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(600):
    if server_proc.poll() is not None:
        print(open("server.log").read())
        raise RuntimeError("เซิร์ฟเวอร์ปิดตัวเอง ดูข้อความด้านบน")
    try:
        print(urllib.request.urlopen("http://127.0.0.1:8000/health").read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("รอเซิร์ฟเวอร์นานเกินไป ดู server.log")

## 6. ทดสอบในเครื่อง

สร้างเสียงพูดภาษาไทย แล้วให้ระบบฟังกลับเป็นข้อความ และถามโมเดลหนึ่งคำถาม

In [ ]:
import json, urllib.request, uuid

def call(path, data, content_type):
    req = urllib.request.Request(f"http://127.0.0.1:8000/v1{path}", data=data, headers={
        "Authorization": f"Bearer {API_KEY}", "Content-Type": content_type})
    return urllib.request.urlopen(req, timeout=300).read()

mp3 = call("/audio/speech", json.dumps({"input": "สวัสดีครับ วันนี้อากาศดีมาก"}).encode(), "application/json")
print("TTS ได้ไฟล์เสียง", len(mp3), "ไบต์")

boundary = uuid.uuid4().hex
form = (f"--{boundary}\r\nContent-Disposition: form-data; name=\"file\"; filename=\"t.mp3\"\r\n"
        f"Content-Type: audio/mpeg\r\n\r\n").encode() + mp3 + f"\r\n--{boundary}--\r\n".encode()
print("STT:", json.loads(call("/audio/transcriptions", form, f"multipart/form-data; boundary={boundary}"))["text"])

reply = json.loads(call("/chat/completions", json.dumps(
    {"messages": [{"role": "user", "content": "แนะนำตัวสั้นๆ หน่อย"}]}).encode(), "application/json"))
print("LLM:", reply["choices"][0]["message"]["content"])

## 7. เปิดลิงก์ให้บอทเข้ามาได้

คัดลอกสองบรรทัดที่พิมพ์ออกมาไปใส่ใน `.env` ของบอท

In [ ]:
import re

tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                               stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if match:
        url = match.group(0)
        break
    time.sleep(1)
if not url:
    raise RuntimeError("สร้างลิงก์ไม่สำเร็จ ดู tunnel.log")

# ลิงก์ใหม่ใช้เวลาสักพักกว่าจะเข้าได้
for _ in range(30):
    try:
        urllib.request.urlopen(f"{url}/health", timeout=10)
        break
    except Exception:
        time.sleep(2)

print("ใส่สองบรรทัดนี้ในไฟล์ .env ของบอท:\n")
print(f"OPENAI_BASE_URL={url}/v1")
print(f"OPENAI_API_KEY={API_KEY}")

## ดูบันทึกการทำงาน

รันเซลล์นี้เมื่อบอทมีปัญหา เพื่อดูว่าเซิร์ฟเวอร์ได้รับคำขออะไรบ้าง

In [ ]:
!tail -n 30 server.log